In [ ]:
# Imports and data-loading block
import os
import json
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

DATA_DIR = "/context/data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

X_PATH = os.path.join(DATA_DIR, "X.parquet")
Y_PATH = os.path.join(DATA_DIR, "y.parquet")
SPLITS_PATH = os.path.join(DATA_DIR, "moons_split.json")

X = pd.read_parquet(X_PATH)
y = pd.read_parquet(Y_PATH)
with open(SPLITS_PATH, "r") as f:
    splits = json.load(f)

X_train = X[X["moon"].isin(splits["train"]) ]
y_train = y[y["moon"].isin(splits["train"]) ]
X_valid = X[X["moon"].isin(splits["reduced_local"]) ]
y_valid = y[y["moon"].isin(splits["reduced_local"]) ]

print("DATA_DIR:", DATA_DIR)
print("X", X.shape, "y", y.shape)
print("train moons", len(splits["train"]), "valid moons", len(splits["reduced_local"]))

In [ ]:
# Train block
import joblib

def train(X_train, y_train, model_dir="./model_crunch"):
    os.makedirs(model_dir, exist_ok=True)
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"]).dropna(subset=["target"])
    df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

    X_fit = df[feature_cols]
    y_fit = df["target"]

    model = LGBMRegressor(
        n_estimators=250,
        learning_rate=0.04,
        num_leaves=128,
        subsample=0.75,
        colsample_bytree=0.7,
        random_state=42,
    )
    model.fit(X_fit, y_fit)

    model_path = os.path.join(model_dir, "model.joblib")
    joblib.dump({"model": model, "features": feature_cols}, model_path)
    print("Saved model to", model_path)
    return model_path

model_path = train(X_train, y_train)

In [ ]:
# Inference block
import joblib

def infer(X_test, model_path):
    obj = joblib.load(model_path)
    model = obj["model"]
    features = obj["features"]

    X = X_test.copy().fillna(0)
    predictions = model.predict(X[features])
    return pd.DataFrame({
        "moon": X["moon"].values,
        "id": X["id"].values,
        "prediction": predictions,
    })

predictions = infer(X_valid, model_path)
print("Predictions shape:", predictions.shape)
print(predictions.head())

for moon in sorted(predictions["moon"].unique()):
    pred = predictions[predictions["moon"] == moon]
    truth = y_valid[y_valid["moon"] == moon]
    merged = pred.merge(truth, on=["moon", "id"])
    corr = pearsonr(merged["prediction"], merged["target"])[0] if len(merged) > 1 else np.nan
    print(f"Moon {moon}: Pearson r = {corr:.4f}")

In [ ]:
# Submission/results block
submission = predictions.copy()
submission_file = "crunch_benchmark_submission.csv"
submission.to_csv(submission_file, index=False)
print("Saved submission to", submission_file)
submission.head()